In [ ]:
# -*- coding: utf-8 -*-
"""
Batch Inference Script for DeepJSCC (Multi SNR and Ratio)
"""
import os
import glob
import torch
from PIL import Image
from torchvision import transforms

from modelEqualization import DeepJSCC, ratio2filtersize
from utils import image_normalization

# -------- CONFIGURATION --------
BASE_INPUT_DIR = 'Root directory with subfolders for input' 
BASE_OUTPUT_DIR = 'Root for saving output'

DEVICE = 'cuda:0' if torch.cuda.is_available() else 'cpu'
DATASET = 'imagenet'
CHANNEL_TYPE = 'Rician'

EQUALIZATION_FLAG = 0

IMAGE_SIZE = (64, 64)

SNR_LIST = [8.0, 4.0, 0.0, -1.0, -2.0, -3.0, -4.0, -5.0, -6.0, -7.0, -8.0]             # Add more SNRs here if needed
RATIO_LIST = [1/4, 1/8, 1/12, 1/16, 1/20, 1/40]         # Add more ratios here if needed


# -------- Auto Checkpoint Finder --------
def auto_find_checkpoint(dataset, c, snr, ratio, channel, equalization_flag, base_dir='./out/checkpoints'):
    prefix = f"{dataset.upper()}_{c}_{snr}_{ratio:.2f}_{channel}_{equalization_flag}"
    candidates = [
        os.path.join(base_dir, d)
        for d in os.listdir(base_dir)
        if os.path.isdir(os.path.join(base_dir, d)) and d.startswith(prefix)
    ]
    if not candidates:
        raise FileNotFoundError(f"No checkpoint directories found with prefix: {prefix}")
    latest_dir = max(candidates, key=os.path.getmtime)
    ckpts = glob.glob(os.path.join(latest_dir, 'epoch_*.pth'))
    if not ckpts:
        raise FileNotFoundError(f"No checkpoint files in: {latest_dir}")
    latest_ckpt = sorted(ckpts, key=os.path.getmtime)[-1]
    print(f"Found checkpoint: {latest_ckpt}")
    return latest_ckpt


# -------- Image Processing --------
def load_image(image_path, image_size):
    transform = transforms.Compose([
        transforms.Resize(image_size),
        transforms.ToTensor(),
    ])
    img = Image.open(image_path).convert("RGB")
    img_tensor = transform(img).unsqueeze(0)
    return img_tensor


# -------- Model Loader --------
def load_model(checkpoint_path, snr, ratio, channel_type, image_size, equalization_flag, device):
    dummy_img = torch.randn(3, *image_size)
    c = ratio2filtersize(dummy_img, ratio)
    print(f"Loading model with inner channel c={c}")

    model = DeepJSCC(c=c, snr=snr, channel_type=channel_type, equalization_flag=equalization_flag)
    model.load_state_dict(torch.load(checkpoint_path, map_location=device))
    model.to(device)
    model.eval()
    return model


# -------- Inference Function --------
def run_inference(model, image_tensor, device):
    image_tensor = image_tensor.to(device)
    with torch.no_grad():
        output = model(image_tensor)
        output = image_normalization('denormalization')(output)
    return output.squeeze(0).cpu()


# -------- Batch Evaluation --------
def process_folder(input_dir, output_dir, model, image_size, device):
    os.makedirs(output_dir, exist_ok=True)

    image_paths = sorted(
        glob.glob(os.path.join(input_dir, '**', '*.*'), recursive=True)
    )
    image_paths = [
        p for p in image_paths if p.lower().endswith(('.jpg', '.jpeg', '.png'))
    ]

    if not image_paths:
        print(f"No image files found in {input_dir}")
        return

    for img_path in image_paths:
        try:
            img_tensor = load_image(img_path, image_size)
            output_tensor = run_inference(model, img_tensor, device)
            output_tensor = output_tensor / 255

            relative_path = os.path.relpath(img_path, input_dir)
            relative_path = os.path.splitext(relative_path)[0] + ".jpg"  # force .png extension
            save_path = os.path.join(output_dir, relative_path)
            os.makedirs(os.path.dirname(save_path), exist_ok=True)

            out_img = transforms.ToPILImage()(output_tensor.clamp(0, 1))
            out_img.save(save_path, format="PNG")
            
            # print(f"Processed: {relative_path}")
        except Exception as e:
            print(f"Failed on {img_path}: {e}")


# -------- Main Loop --------
def main():
    print("Starting batch inference...")

    for snr in SNR_LIST:
        for ratio in RATIO_LIST:
            print(f"\n🔧 SNR: {snr}, Ratio: {ratio:.4f}")

            dummy_img = torch.randn(3, *IMAGE_SIZE)
            c = ratio2filtersize(dummy_img, ratio)

            # Auto checkpoint
            checkpoint_path = auto_find_checkpoint(DATASET, c, snr, ratio, CHANNEL_TYPE,  EQUALIZATION_FLAG)

            # Load model
            model = load_model(checkpoint_path, snr, ratio, CHANNEL_TYPE, IMAGE_SIZE, EQUALIZATION_FLAG, DEVICE)

            # Build input/output paths
            input_dir = BASE_INPUT_DIR
            output_dir = os.path.join(BASE_OUTPUT_DIR, f"Sentry_Data_snr{snr}_ratio{int(1/ratio)}")

            print(f"Input: {input_dir}")
            print(f"Output: {output_dir}")

            # Process
            process_folder(input_dir, output_dir, model, IMAGE_SIZE, DEVICE)

    print("\nAll configurations processed.")


if __name__ == "__main__":
    main()
